## Addition Interp
**"How do small transformers trained on addition data learn to add?"**

authored by : vorrjjard

In [20]:
import torch as t
from torch.utils.data import Dataset, DataLoader, random_split

import matplotlib.pyplot as plt

import transformer_lens

from transformer_lens import HookedTransformer, ActivationCache, HookedTransformerConfig
from transformer_lens.train import HookedTransformerTrainConfig, train

import typing
from typing import Literal, List

from dataclasses import dataclass

import numpy as np

import plotly.express as px

In [2]:
def pad(num: int, reversed: bool=False) -> str:
    num_str = str(num)
    while len(num_str) < 4:
        num_str = '0' + num_str
        
    if reversed:
            num_str = num_str[::-1]  

    return num_str

def generate_sample(cfg) -> list:
    max = cfg.max_output

    d1 = np.random.randint(cfg.min_output, cfg.max_output)
    d2 = np.random.randint(cfg.min_output, cfg.max_output - d1)

    sum = d1 + d2

    return f'{pad(d1)}+{pad(d2)}={pad(sum, reversed=True)}'

In [3]:
DEVICE = 'cuda' if t.cuda.is_available() else 'cpu'
print(DEVICE)

cpu


In [4]:
@dataclass
class dataConfig:
    min_output: int = 0
    n_digits_input = 3
    n_digits_output = 4
    max_output: int = 5000 
    n_samples: int = 50000
data_cfg = dataConfig()
print(data_cfg)

dataConfig(min_output=0, max_output=5000, n_samples=50000)


In [5]:
vocab = {str(i): i for i in range(10)}
vocab['='] = 10
vocab['+'] = 11

In [6]:
vocab_inv = {i : str(i) for i in range(10)}
vocab_inv[10] = '='
vocab_inv[11] = '+'

Let's define a small vocabulary for our project. Hence, our `d_vocab = 10` 

In [7]:
# Generate samples, tokenize, and stack on top of a new batch dimension.

samples_tensor = t.stack(
    [t.tensor([vocab[char] for char in generate_sample(data_cfg)]) for n in range(0, data_cfg.n_samples)]
    ,dim=0)

In [8]:
class SumDataset(Dataset):
    def __init__(self : Dataset, token_dict : dict, samples):
        self.token_dict = token_dict
        self.samples = samples

    def __getitem__(self, idx):
        return {"tokens": self.samples[idx, :]}
    
    def __len__(self):
        return self.samples.shape[0]

In [28]:
ds = SumDataset(vocab, samples_tensor)
cutoffs = [0.8, 0.2]
generator = t.Generator().manual_seed(42)

train_ds, val_ds = random_split(ds, cutoffs, generator=generator)

train_dl = DataLoader(train_ds, batch_size=128, shuffle=True, drop_last=True)
val_dl = DataLoader(val_ds, batch_size=128, shuffle=True, drop_last=True)

##### 2. Model Configuration / Training

In [29]:
def create_model(
    num_digits: int,
    seed: int,
    d_model: int,
    d_head: int,
    n_layers: int,
    n_heads: int,
    d_mlp: int | None,
    normalization_type: str | None,
    device: str = "cuda",
    **kwargs,  # ignore other kwargs
) -> HookedTransformer:
    t.manual_seed(seed)
    np.random.seed(seed)

    attn_only = d_mlp is None

    cfg = HookedTransformerConfig(
        n_layers=n_layers,
        n_ctx=num_digits * 3 + 2,
        d_model=d_model,
        d_head=d_head,
        n_heads=n_heads,
        d_mlp=d_mlp,
        attn_only=attn_only,
        act_fn="relu",
        d_vocab=12,
        use_attn_result=True,
        use_split_qkv_input=True,
        use_hook_tokens=True,

        normalization_type=normalization_type,
        device=device,
    )

    model = HookedTransformer(cfg)
    return model

In [30]:
model = create_model(
    num_digits=4,
    seed=0,
    d_model=48,
    d_head=24,
    n_layers=2,
    n_heads=3,
    normalization_type="LN",
    d_mlp=None,
    device=DEVICE,
)

In [31]:
opt = t.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2, betas=(0.9, 0.98))

##### Training Stuff 

In [34]:
ANS = slice(9, 13)

def masked_loss(logits, tokens):
    logp = logits[:, ANS].log_softmax(-1)
    tgt  = tokens[:, 10:14]
    return -logp.gather(-1, tgt.unsqueeze(-1)).mean()

@t.no_grad()
def accuracy(model, tokens, bs=1024):
    digit, full, n = 0, 0, 0
    for i in range(0, len(tokens), bs):
        b = tokens[i:i+bs]['tokens'].to(DEVICE)
        pred = model(b)[:, ANS].argmax(-1)
        eq = (pred == b[:, 10:14])
        digit += eq.sum().item(); full += eq.all(-1).sum().item(); n += len(b)
    return digit / (n*4), full / n

In [36]:
EPOCHS = 20

for epoch in range(EPOCHS):
    model.train()
    for batch in train_dl:
        tokens = batch["tokens"].to(DEVICE)
        loss = masked_loss(model(tokens), tokens)
        loss.backward(); opt.step(); opt.zero_grad()
    model.eval()
    da, fa = accuracy(model, val_ds)
    print(f"epoch {epoch:2d} | loss {loss.item():.4f} | test digit {da:.3f} full {fa:.3f}")

t.save(model.state_dict(), "../models/sum_model.pt")

epoch  0 | loss 0.0009 | test digit 0.990 full 0.959
epoch  1 | loss 0.0006 | test digit 1.000 full 1.000
epoch  2 | loss 0.0002 | test digit 1.000 full 1.000
epoch  3 | loss 0.0002 | test digit 1.000 full 1.000
epoch  4 | loss 0.0107 | test digit 0.998 full 0.993
epoch  5 | loss 0.0086 | test digit 0.996 full 0.986
epoch  6 | loss 0.0033 | test digit 0.998 full 0.992
epoch  7 | loss 0.0060 | test digit 0.999 full 0.997
epoch  8 | loss 0.0020 | test digit 0.999 full 0.995
epoch  9 | loss 0.0002 | test digit 1.000 full 1.000
epoch 10 | loss 0.0015 | test digit 0.999 full 0.996
epoch 11 | loss 0.0019 | test digit 1.000 full 0.999
epoch 12 | loss 0.0001 | test digit 1.000 full 1.000
epoch 13 | loss 0.0044 | test digit 0.998 full 0.994
epoch 14 | loss 0.0010 | test digit 1.000 full 1.000
epoch 15 | loss 0.0001 | test digit 1.000 full 1.000
epoch 16 | loss 0.0008 | test digit 1.000 full 1.000
epoch 17 | loss 0.0001 | test digit 1.000 full 1.000
epoch 18 | loss 0.0001 | test digit 1.000 full

Training

#### 3. Actual Interp

In [37]:
tokens = samples_tensor[3, :]
tokens

tensor([ 2,  0,  7,  5, 11,  1,  4,  3,  2, 10,  7,  0,  5,  3])

In [38]:

token_labels = [vocab_inv[token.item()] for token in tokens]

logits, cache = model.run_with_cache(tokens.to(DEVICE))

In [39]:
attn_scores_0 = cache['blocks.0.attn.hook_pattern']
attn_scores_1 = cache['blocks.1.attn.hook_pattern']

attn_scores = t.cat([attn_scores_0, attn_scores_1], dim=0)

In [ ]:
sum_scores = attn_scores[:, :, -5:-1, :]

pos_labels = [f"[{i}]{label}" for i, label in enumerate(token_labels)]

px.imshow(
    sum_scores.detach().cpu().numpy(),
    facet_col=0,
    facet_row=1,
    labels=dict(x='Key', y='Query'),
    x=pos_labels,
    y=pos_labels[-5:-1],
    zmin=0,
    zmax=1,
    color_continuous_scale=[[0, 'black'], [1, '#fde725']],
)